In [ ]:
from pathlib import Path
import runpy

def _find_notebook_bootstrap(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "notebooks" / "_bootstrap.py"
        if direct.exists():
            return direct
        nested = candidate / "abstractgraph-ml" / "notebooks" / "_bootstrap.py"
        if nested.exists():
            return nested
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_find_notebook_bootstrap(Path.cwd())))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from abstractgraph.utils import plot_dataset_method_bars, plot_pareto
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

# NeuralGraphEstimator Demo

This notebook demonstrates training the Transformer-based classifier on synthetic graphs using the node-level vectorizer.

In [ ]:
import numpy as np
import networkx as nx
import torch
from sklearn.metrics import accuracy_score

from abstractgraph.vectorize import AbstractGraphNodeTransformer
from abstractgraph_ml.neural import NeuralGraphEstimator
from abstractgraph.operators import *

# Suppress PyTorch Transformer nested tensor warning in this notebook
import warnings
warnings.filterwarnings(
    'ignore',
    message=r'.*enable_nested_tensor is True, but self.use_nested_tensor is False.*',
    category=UserWarning,
    module='torch.nn.modules.transformer'
)

## UMAP visualization of test embeddings

In [ ]:
import time
from sklearn.metrics import roc_auc_score, average_precision_score
from abstractgraph.utils import plot_embedding_2d
def perf(neural_graph_estimator, graphs, targets):    
    # Train metrics
    t_pred0 = time.perf_counter()
    pred = neural_graph_estimator.predict(graphs)
    try:
        proba = neural_graph_estimator.predict_proba(graphs)
    except Exception:
        proba = None
    t_pred1 = time.perf_counter()
    pred_time = t_pred1 - t_pred0
    acc = accuracy_score(targets, pred)
    errors = int((pred != targets).sum())
    if proba is not None and proba.ndim == 2 and proba.shape[1] > 1:
        if proba.shape[1] == 2:
            auc = roc_auc_score(targets, proba[:, 1])
            ap = average_precision_score(targets, proba[:, 1])
        else:
            auc = roc_auc_score(targets, proba, average='macro', multi_class='ovr')
            ap = average_precision_score(targets, proba, average='macro')
    else:
        auc, ap = None, None
    return acc, auc, ap, errors, pred_time

def _compute_embeddings(graph_estimator, graphs):
    try:
        import umap.umap_ as umap

        embs = graph_estimator.transform(graphs)
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message=r".*n_jobs value .* overridden .* by setting random_state.*",
                category=UserWarning,
                module="umap.umap_",
            )
            Z = umap.UMAP(n_components=2, random_state=42, init="random").fit_transform(embs)
        return Z
    except Exception:
        return None
    

def plot(graph_estimator, graphs, targets, graphs_test, targets_test):
    alpha=.65
    size = 6
    embeddings = _compute_embeddings(graph_estimator, graphs)
    fig, axes = plt.subplots(1, 3, figsize=(3.1*size, size), constrained_layout=True)
    _ = plot_embedding_2d(graph_estimator, graphs, targets, title_prefix="Train embeddings", mode="scatter", alpha=alpha, ax=axes[0], show=False, Z=embeddings)
    _ = plot_embedding_2d(graph_estimator, graphs, targets, title_prefix="Train embeddings", mode="knn_class_union", k=5, alpha=alpha, z=1, ax=axes[1], show=False, show_instances=False, Z=embeddings)
    _ = plot_embedding_2d(graph_estimator, graphs, targets, title_prefix="Train embeddings", mode="knn_class_union", k=11, alpha=alpha, z=1, ax=axes[2], show=False, show_instances=False, Z=embeddings)
    plt.show()

    embeddings = _compute_embeddings(graph_estimator, graphs_test)
    fig, axes = plt.subplots(1, 3, figsize=(3.1*size, size), constrained_layout=True)
    _ = plot_embedding_2d(graph_estimator, graphs_test, targets_test, title_prefix="Test embeddings", mode="scatter", alpha=alpha, ax=axes[0], show=False, Z=embeddings)
    _ = plot_embedding_2d(graph_estimator, graphs_test, targets_test, title_prefix="Test embeddings", mode="knn_class_union", k=5, alpha=alpha, z=1, ax=axes[1], show=False, show_instances=False, Z=embeddings)
    _ = plot_embedding_2d(graph_estimator, graphs_test, targets_test, title_prefix="Test embeddings", mode="knn_class_union", k=11, alpha=alpha, z=1, ax=axes[2], show=False, show_instances=False, Z=embeddings)
    plt.show()

def print_perf(header_str, acc, auc, ap, errors, fit_time, pred_time, n_instances):
    n_instances_per_second = n_instances / (fit_time+pred_time)
    print(f"{header_str}: accuracy={acc:.3f}, roc_auc={(auc if auc is not None else float('nan')):.3f}, avg_precision={(ap if ap is not None else float('nan')):.3f}, errors={errors}, fit_time={fit_time:.2f}s, pred_time={pred_time:.2f}s, total_time={(fit_time+pred_time):.2f}s, n_instances_per_second={n_instances_per_second:.1f}  ")

## Dataset info & examples

In [ ]:
from abstractgraph_graphicalizer.chem import PubChemLoader

loader = PubChemLoader(on_error="skip")
#assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_ids = ['492992','488975']
#assay_ids = ['488975'] #active 2,634
datasets = []
for assay_id in assay_ids:
            original_graphs, original_targets = loader.load(assay_id)
    print(f'AID{assay_id}  #graphs: {len(original_graphs)}')
    datasets.append((assay_id, original_graphs, original_targets))

In [ ]:
nbits = 14

cycle_and_tree = (cycle(), compose(neighborhood(radius=1), tree()))
pre_train_df = add(*cycle_and_tree, compose_product(binary_combination(distance=0), *cycle_and_tree))
pre_train_df = compose(pre_train_df, unlabel())

def make_nsppk(radius=1, distance=3, connector=1):
    if connector == 0:
        df = add(union_of_shortest_paths(length=distance), compose(combination(number_of_elements=2,distance=(0,distance)), neighborhood(radius=(0,radius))))
    elif connector == 1:
        df = compose(combination(number_of_elements=2,distance=(0,distance)), neighborhood(radius=(0,radius)))
    else:
        raise Exception(f'connector {connector} is not allowed')
    return df
df = make_nsppk(radius=1, distance=4, connector=1)

In [ ]:
from abstractgraph.display import display_decomposition_graph
print('graph base decomposition')
display_decomposition_graph(df)

print('graph domain pre-training')
display_decomposition_graph(pre_train_df)

In [ ]:
from abstractgraph.vectorize import AbstractGraphTransformer
transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=df,
    return_dense=True,
    n_jobs=-1,
)

from sklearn.ensemble import RandomForestClassifier
from abstractgraph_ml.estimators import GraphEstimator
graph_estimator = GraphEstimator(
    transformer=transformer,
    estimator=RandomForestClassifier(random_state=0, n_estimators=300, n_jobs=-1),
    manifold=None,
    n_selected_features=500,
)

In [ ]:
from abstractgraph.vectorize import AbstractGraphNodeTransformer
node_vectorizer = AbstractGraphNodeTransformer(nbits=nbits, decomposition_function=df, return_dense=True, n_jobs=-1)

from abstractgraph_ml.neural import NeuralGraphEstimator, InputAdapterFactorized

# Infer the input dimension from one sample graph to build a custom adapter.
sample_graph = datasets[0][1][0]
sample_mat = node_vectorizer.transform([sample_graph])[0]
input_dim = int(sample_mat.shape[1])

d_model = 2**8
adapter_bottleneck = 2**7
adapter = InputAdapterFactorized(
    in_dim=input_dim,
    d_model=d_model,
    bottleneck=adapter_bottleneck,
    first_bias=False,
    second_bias=True,
)
adapter.svd_init(input_dim, d_model, adapter_bottleneck)

neural_graph_estimator = NeuralGraphEstimator(
    # Vectorization
    node_vectorizer=node_vectorizer,              # node-to-feature transformer; yields per-node dense features

    # Input adapter
    adapter=adapter,                              # custom adapter module (modular override)

    # Model size
    d_model=d_model,                             # Transformer model dimension
    nhead=4,                                      # number of self-attention heads
    num_layers=2,                                 # number of Transformer encoder layers
    dim_feedforward=2**9,                         # hidden size of the encoder feed-forward network

    # Regularization and activations
    dropout=0.25,                                 # dropout probability in encoder blocks
    activation='gelu',                            # activation function inside Transformer layers
    pooling='cls',                                # graph embedding via learnable [CLS] token ('mean'|'max'|'cls')

    # Training schedule
    epochs=100,                                    # training epochs
    batch_size=16,                                # graphs per batch; nodes are padded within each batch
    val_split=0.1,                                # fraction held out for validation (enables early stopping)
    early_stopping_patience=3,                    # epochs without val improvement before stopping
    seed=4213,                                    # random seed for reproducibility

    # Optimizer
    lr=1e-3,                                      # AdamW learning rate
    weight_decay=1e-4,                            # AdamW weight decay

    # Auxiliary losses
    aux_whitening_weight=1e-3,                    # strength of whitening loss on proj1 activations (e.g., 1e-3)
    aux_sparsity_w2_weight=1e-5,                  # L1 penalty on proj2 weights W2 for sparsity (e.g., 1e-5)

    # Logging
    verbose=True,                                 # print metrics and parameter counts each epoch

    # LoRA configuration (for fine-tuning)
    lora_r=8,                                     # LoRA rank
    lora_alpha=16.0,                              # LoRA scaling factor
    lora_dropout=0.1,                             # dropout on LoRA updates
    lora_scope="encoder_only",                    # where LoRA is applied "encoder_only", "head_only", "ffn_only", "adapter_only", "all_linear"
)


In [ ]:
from sklearn.model_selection import train_test_split
import time

assay_id, graphs, targets = datasets[0]
pre_train_assay_id, pre_train_graphs, pre_train_targets = datasets[1]
n_instances = len(graphs)
targets = np.array(targets)
# Train/test split (scikit-learn)
graphs_tr, graphs_te, targets_tr, targets_te = train_test_split(graphs, targets, test_size=0.2, random_state=0)
graphs_tr, graphs_ft, targets_tr, targets_ft = train_test_split(graphs_tr, targets_tr, test_size=0.5, random_state=0)

print('_'*100)
print(f'assay_id:{assay_id}   #graphs:{n_instances}   #train:{len(graphs_tr)}   #fine-tune:{len(graphs_ft)}   #test:{len(graphs_te)}')
print(f'pre_train_assay_id:{pre_train_assay_id}   #graphs:{len(pre_train_graphs)}')

print('Decomposition Graph Kernel Estimator')
t_fit0 = time.perf_counter()
graph_estimator.fit(graphs_tr, targets_tr)
t_fit1 = time.perf_counter()
fit_time = t_fit1 - t_fit0
acc_te, auc_te, ap_te, errors_te, pred_time_te = perf(graph_estimator, graphs_te, targets_te)
print_perf('Test ', acc_te, auc_te, ap_te, errors_te, fit_time, pred_time_te, len(graphs))
plot(graph_estimator, graphs, targets, graphs_te, targets_te)

print('Neural Graph Estimator')
t_fit0 = time.perf_counter()
neural_graph_estimator.fit(graphs_tr, targets_tr)
t_fit1 = time.perf_counter()
fit_time = t_fit1 - t_fit0
acc_te, auc_te, ap_te, errors_te, pred_time_te = perf(neural_graph_estimator, graphs_te, targets_te)
print_perf('Test ', acc_te, auc_te, ap_te, errors_te, fit_time, pred_time_te, len(graphs))
plot(neural_graph_estimator, graphs, targets, graphs_te, targets_te)

print('Neural Graph Estimator with additional fine-tuning')
t_fit0 = time.perf_counter()
neural_graph_estimator.lora_scope="encoder_only"
neural_graph_estimator.fine_tune(graphs_ft, targets_ft)
t_fit1 = time.perf_counter()
fit_time = t_fit1 - t_fit0
acc_te, auc_te, ap_te, errors_te, pred_time_te = perf(neural_graph_estimator, graphs_te, targets_te)
print_perf('Test ', acc_te, auc_te, ap_te, errors_te, fit_time, pred_time_te, len(graphs))
plot(neural_graph_estimator, graphs, targets, graphs_te, targets_te)

print('Neural Graph Estimator with pre-training and fine-tuning')
t_fit0 = time.perf_counter()
neural_graph_estimator.reset() #RESET
neural_graph_estimator.lora_scope="all_linear"
neural_graph_estimator.pre_train(pre_train_graphs, decomposition_function=pre_train_df, nbits=nbits)
neural_graph_estimator.fine_tune(graphs_ft, targets_ft)
t_fit1 = time.perf_counter()
fit_time = t_fit1 - t_fit0
acc_te, auc_te, ap_te, errors_te, pred_time_te = perf(neural_graph_estimator, graphs_te, targets_te)
print_perf('Test ', acc_te, auc_te, ap_te, errors_te, fit_time, pred_time_te, len(graphs))
plot(neural_graph_estimator, graphs, targets, graphs_te, targets_te)

print('Neural Graph Estimator with pre-training and fit')
t_fit0 = time.perf_counter()
neural_graph_estimator.reset() #RESET
neural_graph_estimator.lora_scope="encoder_only"
neural_graph_estimator.pre_train(pre_train_graphs, decomposition_function=pre_train_df, nbits=nbits)
neural_graph_estimator.fit(graphs_tr, targets_tr)
neural_graph_estimator.fine_tune(graphs_ft, targets_ft)
t_fit1 = time.perf_counter()
fit_time = t_fit1 - t_fit0
acc_te, auc_te, ap_te, errors_te, pred_time_te = perf(neural_graph_estimator, graphs_te, targets_te)
print_perf('Test ', acc_te, auc_te, ap_te, errors_te, fit_time, pred_time_te, len(graphs))
plot(neural_graph_estimator, graphs, targets, graphs_te, targets_te)


---